# 多随机种子全轮次模型性能分析

# 1 公共参数与数据准备

## 1.1 公共参数与数据读取

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.metrics import make_scorer, mean_absolute_error, r2_score
from sklearn.model_selection import KFold, RandomizedSearchCV
from xgboost import XGBRegressor
from IPython.display import clear_output


# 设置正式输入、输出和统一特征顺序；案例运行仅替换此参数块。
PROJECT_ROOT = Path.cwd()

DATA_PATH = PROJECT_ROOT / "Example" / "ALL_data_analysis" / "ALL_test.csv" #ALL_test.csv  ALL_exp._results.csv
OUTPUT_PATH = PROJECT_ROOT / "Example" / "ALL_data_analysis" / "model_analysis"

ANALYSIS_ROUND = ["R1","R2","R3","R4"] 

all_data = pd.read_csv(DATA_PATH)
input_files = {}
for round in ANALYSIS_ROUND:
    input_files[round] = all_data[all_data["remarks"]==round]


# 统一目标与特征。 'H'
TARGET_COLUMN = "titer(mg/L)"
FEATURE_COLUMNS = ['(NH4)2SO4','Triton X-100','Glycine','H',
                    'CSL-P','NaCl','K2HPO4','Tryptone',
                    'YE','Methionine','Cysteine','NH4OAc',
                    'Glycerol','Na2S2O3','ZnSO4·7H2O','MgSO4·7H2O','FAC']  
# 以列表形式设置任意数量的随机种子，并设置模型筛选口径和并行计算参数。
random_seeds = [0, 1, 42, 123, 256, 512, 1024, 2023, 2024, 2025]
# test ：  [0, 1][0, 1, 42, 123, 256, 512, 1024, 2023, 2024, 2025]

ensemble_size = 20
search_iterations = 200
cv_fold_count = 5

parameter_distributions = {
    "learning_rate": [0.01, 0.03, 0.1, 0.3],
    "colsample_bytree": [0.6, 0.8, 0.9, 1.0],
    "subsample": [0.6, 0.8, 0.9, 1.0],
    "max_depth": [2, 3, 4, 6, 8],
    "n_estimators": [10, 20, 40, 60, 80, 100, 300, 500],
    "reg_lambda": [1, 1.5, 2],
    "gamma": [0, 0.1, 0.4, 0.6],
    "min_child_weight": [1, 2, 4],
}

# 创建输出目录
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# 读取每轮数据，校验工作表和必需字段，并统一为17个体积特征、目标值和来源行号。
round_data = {}
source_records = []
for round_name, source_frame in input_files.items():

    required_columns = FEATURE_COLUMNS + [TARGET_COLUMN]

    round_frame = source_frame[required_columns].copy()

    round_frame.insert(0, "source_row", np.arange(2, len(round_frame) + 2))
    round_frame.insert(0, "round", round_name)
    round_data[round_name] = round_frame
    source_records.append({
        "round": round_name,
        "input_path": str(DATA_PATH),
        "modified_time": pd.Timestamp(DATA_PATH.stat().st_mtime, unit="s"),
        "row_count": len(source_frame),
        "column_count": source_frame.shape[1],
        "field_names": ", ".join(source_frame.columns.astype(str)),
    })

# 显示影响结果的核心参数和输入溯源信息。
run_parameters = pd.DataFrame({
    "parameter": [
        "random_seeds", "seed_count", "search_iterations", "cv_fold_count", "ensemble_size", "evaluation_round",
    ],
    "value": [
        str(random_seeds), len(random_seeds), search_iterations, cv_fold_count, ensemble_size, "round4",
    ],
})
source_info = pd.DataFrame(source_records)
display(run_parameters)
display(source_info)

,parameter,value
0,random_seeds,"[0, 1]"
1,seed_count,2
2,search_iterations,200
3,cv_fold_count,5
4,ensemble_size,20
5,evaluation_round,round4


,round,input_path,modified_time,row_count,column_count,field_names
0,R1,c:\Users\Liaoyt\Desktop\Ai_medium\github\Examp...,2026-09-17 06:19:53.150386572,91,20,"(NH4)2SO4, Triton X-100, Glycine, H, CSL-P, Na..."
1,R2,c:\Users\Liaoyt\Desktop\Ai_medium\github\Examp...,2026-09-17 06:19:53.150386572,90,20,"(NH4)2SO4, Triton X-100, Glycine, H, CSL-P, Na..."
2,R3,c:\Users\Liaoyt\Desktop\Ai_medium\github\Examp...,2026-09-17 06:19:53.150386572,90,20,"(NH4)2SO4, Triton X-100, Glycine, H, CSL-P, Na..."
3,R4,c:\Users\Liaoyt\Desktop\Ai_medium\github\Examp...,2026-09-17 06:19:53.150386572,90,20,"(NH4)2SO4, Triton X-100, Glycine, H, CSL-P, Na..."


# 2 多随机种子的三阶段训练与第四轮评估

依次遍历公共列表 `random_seeds` 中的每个种子，并完成三个累计训练阶段。各阶段在全部累计历史数据上随机抽取 200 组候选参数并执行 5 折交叉验证，按平均 MAE 选择前 20 组；记录入选 20 组参数的平均 R²、Spearman 和 MAE，再以全部累计数据训练 20 个成员并仅预测 Round4。逐种子结果和跨种子汇总均使用动态长表，所有预期行数都由 `len(random_seeds)` 计算。

## 2.1 数据处理

In [6]:
def fit_ensemble(X, y, parameter_list, random_seed):
    """按给定参数组和随机种子训练20成员XGBoost集成。"""
    return [
        XGBRegressor(
            objective="reg:squarederror",
            random_state=random_seed,
            n_jobs=-1,
            **parameters,
        ).fit(X, y)
        for parameters in parameter_list
    ]


def ensemble_predict(models, X):
    """计算集成成员预测的逐样本均值。"""
    return np.mean([model.predict(X) for model in models], axis=0)


def calculate_spearman(actual, predicted):
    """计算真实值与预测值的Spearman秩相关系数。"""
    return float(spearmanr(actual, predicted).statistic)


# 定义三个累计训练阶段和模型筛选指标。
stage_specs = [
    {"model": "Round1", "training_rounds": ["R1"]},
    {"model": "Round1+2", "training_rounds": ["R1", "R2"]},
    {"model": "Round1+2+3", "training_rounds": ["R1", "R2", "R3"]},
]
cv_scoring = {
    "r2": "r2",
    "spearman": make_scorer(calculate_spearman),
    "mae": "neg_mean_absolute_error",
}
metric_records = []

# 统一使用round 4作为测试集
evaluation_data = round_data["R4"]
evaluation_X = evaluation_data[FEATURE_COLUMNS]
evaluation_y = evaluation_data[TARGET_COLUMN].to_numpy()

# 将每个模型自己推荐的样本作为测试集进行评估
next_data = ["R2","R3","R4"]
next_id = 0


# 对随机种子列表中的每个种子和三个累计训练阶段依次执行参数筛选、集成重训及第四轮评估。
for stage in stage_specs:

    next_id += 1 
    
    for random_seed in random_seeds:
        clear_output()
        print("model:{}".format(stage["model"]))
        print(f"random_seed:{random_seed}")
        # 由阶段字典中获取训练集
        training_data = pd.concat(
            [round_data[name] for name in stage["training_rounds"]],
            ignore_index=True,
        )
        X_full = training_data[FEATURE_COLUMNS]
        y_full = training_data[TARGET_COLUMN]

        # 使用同一种子控制参数抽样、交叉验证划分和XGBoost随机过程。
        search = RandomizedSearchCV(
            estimator=XGBRegressor(
                objective="reg:squarederror",
                random_state=random_seed,
                n_jobs=-1,
            ),
            param_distributions=parameter_distributions,
            n_iter=search_iterations,
            scoring=cv_scoring,
            cv=KFold(
                n_splits=cv_fold_count,
                shuffle=True,
                random_state=random_seed,
            ),
            random_state=random_seed,
            n_jobs=-1,
            refit=False,
            error_score="raise",
        )
        search.fit(X_full, y_full)
        search_results = pd.DataFrame(search.cv_results_)

        # 按五折平均MAE选择前20组参数，并计算入选参数的三项指标平均值。
        selected_results = (
            search_results
            .sort_values("mean_test_mae",ascending=False,kind="stable",)
            .head(ensemble_size)
            .copy()
        )

        top_parameters = selected_results["params"].tolist()
        
        selected_top20_cv_r2_mean = selected_results["mean_test_r2"].mean()
        selected_top20_cv_spearman_mean = selected_results["mean_test_spearman"].mean()
        selected_top20_cv_mae_mean = -selected_results["mean_test_mae"].mean()

        # 使用累计训练数据重训20个成员，并仅计算第四轮集成预测及评估指标。
        final_models = fit_ensemble(X_full, y_full, top_parameters, random_seed)
        round4_predicted_yield = ensemble_predict(final_models, evaluation_X)

        evaluation_data = round_data[next_data[next_id-1]]
        next_X = evaluation_data[FEATURE_COLUMNS]
        next_y = evaluation_data[TARGET_COLUMN].to_numpy()

        next_predicted_yield = ensemble_predict(final_models,next_X)

        # 最终结果汇总
        metric_records.append({
            "random_seed": random_seed,
            "model": stage["model"],
            "training_rounds": "+".join(stage["training_rounds"]),

            "selected_top20_cv_r2_mean": selected_top20_cv_r2_mean,
            "selected_top20_cv_spearman_mean": selected_top20_cv_spearman_mean,
            "selected_top20_cv_mae_mean": selected_top20_cv_mae_mean,

            "round4_r2": r2_score(evaluation_y, round4_predicted_yield),
            "round4_spearman": calculate_spearman(evaluation_y, round4_predicted_yield),
            "round4_mae": mean_absolute_error(evaluation_y, round4_predicted_yield),

            "next_r2": r2_score(next_y, next_predicted_yield),
            "next_spearman": calculate_spearman(next_y, next_predicted_yield),
            "next_mae": mean_absolute_error(next_y, next_predicted_yield),

            "training_n": len(training_data),
            "round4_n": len(evaluation_data),
            "cv_fold_count": cv_fold_count,
            "searched_parameter_count": len(search_results),
            "selected_parameter_count": len(selected_results),
            "selection_metric": "cv_mae_mean_ascending",
        })

# 生成每个随机种子、每个模型一行的内部宽表，并校验基础结果粒度。
seed_model_metrics_wide = pd.DataFrame(metric_records)
seed_model_metrics_wide.to_csv(OUTPUT_PATH/"seed_model_metrics_wide.csv")





model:Round1+2+3
random_seed:1


## 2.1 均值与显著性分析

In [7]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings("ignore")

# ===============分数均值±标准误计算=====================
score_list = ["selected_top20_cv_r2_mean",
            "selected_top20_cv_spearman_mean",
            "selected_top20_cv_mae_mean",
            "round4_r2",
            "round4_spearman",
            "round4_mae",
            "next_r2",
            "next_spearman",
            "next_mae",
            ]

record = []
for stage in stage_specs:
    
    score_record = []
    score_record.append(stage["model"])
    for score in score_list:
        scores = seed_model_metrics_wide[seed_model_metrics_wide["model"]==stage["model"]][score]

        n = len(scores)
        mean = np.mean(scores)                  # 均值
        std_sample = np.std(scores, ddof=1)     # 样本标准差
        sem = std_sample / np.sqrt(n)           # 标准误（未使用）

        score_record.append(f"{mean:.4f}±{std_sample:.4f}")

    record.append(score_record)
record = np.array(record)

selected_top20_cv = pd.DataFrame(record[:,0:4],columns=["model", "cv_r2", "cv_spearman","cv_mae"])
round4 = pd.DataFrame(record[:,[0,*range(4,7)]],columns=["model", "round4_r2", "round4_spearman","round4_mae"])
next = pd.DataFrame(record[:,[0,*range(7,10)]],columns=["model", "next_r2", "next_spearman","next_mae"])

display(selected_top20_cv)
display(round4)
display(next)


,model,cv_r2,cv_spearman,cv_mae
0,Round1,0.7437±0.0006,0.8071±0.0075,37.5830±0.6246
1,Round1+2,0.8178±0.0200,0.8712±0.0086,29.3828±0.9919
2,Round1+2+3,0.8479±0.0045,0.8240±0.0064,26.4282±0.4687


,model,round4_r2,round4_spearman,round4_mae
0,Round1,-0.1961±0.1113,0.3615±0.0387,40.3563±2.1505
1,Round1+2,0.3251±0.0102,0.5402±0.0013,28.2941±0.1637
2,Round1+2+3,0.3852±0.0119,0.6022±0.0109,26.4779±0.0750


,model,next_r2,next_spearman,next_mae
0,Round1,0.0967±0.0250,0.4000±0.0256,34.4102±0.1308
1,Round1+2,-0.5918±0.0975,0.3415±0.0141,26.6907±1.0773
2,Round1+2+3,0.3852±0.0119,0.6022±0.0109,26.4779±0.0750
